# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [5]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from groq import Groq

In [6]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('gsk_') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama-3.3-70b-versatile'
openai = Groq(api_key=api_key)

API key looks good so far


In [7]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [11]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog/posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'company facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama-3.3-70b-versatile
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog/posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 6 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'docs', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'learn', 'url': 'https://huggingface.co/learn'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 8 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
nvidia/personaplex-7b-v1
Updated
about 8 hours ago
•
50.8k
•
1.38k
moonshotai/Kimi-K2.5
Updated
about 3 hours ago
•
21.4k
•
952
Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice
Updated
6 days ago
•
169k
•
711
microsoft/VibeVoice-ASR
Updated
2 days ago
•
102k
•
703
Tongyi-MAI/Z-Image
Updated
about 17 hours ago
•
1.21k
•
610
Browse 2M+ models
Spaces
Running
on
Zero
Featured
946
Qwen3-TTS Demo
🎙
946
Generate speech from text with custom voices and cloning
Running
on
Zero
MCP
1.87k
Z Image Turbo
🖼
1.87k
Generate stunning AI images from t

In [18]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 6 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nnvidia/personaplex-7b-v1\nUpdated\nabout 8 hours ago\n•\n50.8k\n•\n1.38k\nmoonshotai/Kimi-K2.5\nUpdated\nabout 3 hours ago\n•\n21.4k\n•\n953\nQwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\nUpdated\n6 days ago\n•\n169k\n•\n711\nmicrosoft/VibeVoice-ASR\nUpdated\n2 days ago\n•\n102k\n•\n703\nTongyi-MAI/Z-Image\nUpdated\nabout 17 hours ago\n•\n1.21k\n•\n610\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 6 relevant links


# Hugging Face Brochure
## Introduction
Hugging Face is the AI community building the future. The platform is where the machine learning community collaborates on models, datasets, and applications. With over 2 million models, 1 million applications, and 500,000 datasets, Hugging Face is the go-to destination for machine learning enthusiasts, researchers, and professionals.

## Our Mission
At Hugging Face, our mission is to create, discover, and collaborate on machine learning better. We provide a platform for the community to host and collaborate on unlimited public models, datasets, and applications. Our open-source stack enables users to move faster and explore all modalities, including text, image, video, audio, and 3D.

## Company Culture
Our company culture is centered around collaboration, innovation, and community. We believe in the power of open-source collaboration and strive to create a platform that is accessible to everyone. Our community is at the heart of everything we do, and we encourage users to share their work, build their portfolio, and learn from others.

## Customers
Our platform is used by a wide range of customers, from individual researchers to large enterprises. Our customers use our platform to build, deploy, and manage their machine learning models, datasets, and applications. We provide a range of tools and services to support our customers, including our enterprise platform, pricing plans, and documentation.

## Careers
We are always looking for talented individuals to join our team. Our careers page lists current openings in areas such as engineering, research, and sales. We offer a dynamic and collaborative work environment, opportunities for professional growth, and a chance to work on cutting-edge machine learning projects.

## Community
Our community is what sets us apart. We have a thriving community of users who contribute to our platform, share their knowledge, and learn from others. Our community blog features articles, tutorials, and research papers on machine learning, natural language processing, computer vision, and more. We also provide a range of resources, including documentation, tutorials, and forums, to support our community.

## Join Us
Whether you are a machine learning enthusiast, a researcher, or a professional, we invite you to join our community and be a part of building the future of AI. Explore our platform, browse our models and datasets, and start building your own machine learning projects today.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 6 relevant links


# Introduction to Hugging Face
Hugging Face is a cutting-edge AI community that is building the future of machine learning. The company provides a platform where developers, researchers, and industry experts collaborate on models, datasets, and applications. With a vast repository of over 2 million models, 1 million applications, and 500,000 datasets, Hugging Face is the go-to destination for anyone looking to push the boundaries of AI.

## Company Culture
At Hugging Face, the culture is built around collaboration, innovation, and community. The company believes in the power of open-source technology and provides a platform for developers to share their work, learn from others, and build upon existing projects. The Hugging Face community is diverse, passionate, and driven by a shared vision of shaping the future of AI.

## Customers and Partnerships
Hugging Face works with a wide range of customers, from individual developers to large enterprises, to help them build and deploy AI models. The company has partnered with industry leaders such as NVIDIA, Microsoft, and Facebook to advance the state-of-the-art in AI research and development. With its open-source stack and collaborative platform, Hugging Face is empowering organizations to move faster and achieve their AI goals.

## Careers and Jobs
Hugging Face is committed to building a talented and diverse team of individuals who share its passion for AI and community. The company offers a range of career opportunities, from engineering and research to sales and marketing. If you're interested in joining a dynamic and innovative team, check out the current openings on the Hugging Face careers page.

## Community and Resources
The Hugging Face community is at the heart of everything the company does. The platform provides a range of resources, including documentation, tutorials, and forums, to help developers get started with AI and machine learning. The company also publishes a regular blog, featuring articles on the latest developments in AI research, industry trends, and community news.

## Join the Hugging Face Community
Whether you're a seasoned AI expert or just starting out, Hugging Face invites you to join its vibrant community of developers, researchers, and industry leaders. With its collaborative platform, open-source technology, and commitment to innovation, Hugging Face is the perfect place to learn, share, and build the future of AI. Sign up today and start exploring the possibilities of machine learning.

In [25]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 7 relevant links


# Introduction to Hugging Face
Hugging Face is a cutting-edge AI company that has created a collaborative platform for the machine learning community. The platform allows users to create, discover, and collaborate on machine learning models, datasets, and applications.

## Company Culture
At Hugging Face, the culture is centered around collaboration, innovation, and community. The company provides a platform for machine learning enthusiasts to share their work, learn from others, and build their portfolios. The platform is open-source, allowing users to host and collaborate on unlimited public models, datasets, and applications.

## Customers
Hugging Face serves a wide range of customers, from individual machine learning enthusiasts to large enterprises. The platform provides a unique opportunity for customers to collaborate with the machine learning community, access a vast library of models and datasets, and build their own machine learning applications.

## Careers
Hugging Face is a dynamic and innovative company that is always looking for talented individuals to join its team. The company offers a range of career opportunities, from machine learning engineering to community management. If you are passionate about machine learning and want to be part of a cutting-edge company, Hugging Face may be the perfect place for you.

## Products and Services
Hugging Face offers a range of products and services, including:

* **Models**: A library of over 2 million machine learning models that can be used for a wide range of applications.
* **Datasets**: A collection of over 500,000 datasets that can be used to train and test machine learning models.
* **Spaces**: A platform for hosting and collaborating on machine learning applications.
* **Community**: A community of machine learning enthusiasts that provides support, feedback, and collaboration opportunities.
* **Enterprise**: A range of enterprise solutions that provide large organizations with the tools and support they need to build and deploy machine learning applications.

## Blog and Community
Hugging Face has a vibrant blog and community that provides insights into the latest developments in machine learning, as well as tutorials, guides, and case studies. The blog features articles on topics such as NLP, computer vision, reinforcement learning, and ethics, and provides a platform for community members to share their knowledge and expertise.

Overall, Hugging Face is a unique and innovative company that is pushing the boundaries of what is possible with machine learning. Whether you are a machine learning enthusiast, a developer, or an enterprise customer, Hugging Face has something to offer.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>